In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_F.xpt"
csv_file = DATA_DIR / "ALQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 ALQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ140Q",
    "ALQ140U",
    "ALQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6059:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 9:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,059 rows x 9 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Do NOT globally replace this value.
#
# First validate:
#
#   1. Which variables contain it
#   2. How many occurrences exist
#   3. Whether CDC permits zero for those variables
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
# 1. Attached ALQ_F.xpt
# 2. CDC 2009-2010 ALQ_F codebook
#
#
# ALQ120Q:
#
# "How often drink alcohol over past 12 mos"
#
# CDC instructions:
#
#     ENTER '0' FOR NEVER
#
#
# ALQ140Q:
#
# "# days have 5 or more drinks/past 12 mos"
#
# CDC instructions:
#
#     ENTER '0' FOR NONE
#
#
# Therefore these tiny values represent legitimate
# numeric zeros.
#

EXPECTED_TINY_COUNTS = {
    "ALQ120Q": 986,
    "ALQ140Q": 2159
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated ALQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 3145:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated ALQ_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY exact TINY_VALUE occurrences in:
#
#     ALQ120Q
#     ALQ140Q
#
# No blanket global replacement.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 3145:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# data SHOULD contain.
#
# It differs from source_df ONLY at the validated
# tiny-zero positions.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This check is performed BEFORE any Int64 conversion.
#
# For ALQ_F there are no genuine fractional values.
#
# If any were found, the script would STOP rather
# than round them.
#
# This same safeguard protects Dietary, Laboratory,
# Examination, and other continuous measurements.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values found in {col}"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded or changed:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in ALQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# All ALQ_F fields contain whole-number questionnaire
# codes/counts after validated zero restoration.
#
# The fractional check above proves conversion is safe
# for THIS file.
#
# NO .round() is used.
#

DATA_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DATA_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        if not np.all(
            np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: ALQ_F questionnaire fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - ALQ101
# ============================================================

#
# CDC:
#
# 1 = Yes          3,906
# 2 = No           1,423
# 7 = Refused          0
# 9 = Don't know       3
# Missing            727
#

EXPECTED_ALQ101 = {
    1: 3906,
    2: 1423,
    9: 3
}


actual = (
    df["ALQ101"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ101 ---")


if (
    actual != EXPECTED_ALQ101
    or
    int(
        df["ALQ101"]
        .isna()
        .sum()
    )
    != 727
):

    raise ValueError(
        "ALQ101 differs from CDC."
    )


print(
    "PASS: ALQ101 matches CDC."
)


# ============================================================
# 12. CDC CHECK - ALQ110
# ============================================================

#
# CDC:
#
# 1 = Yes            746
# 2 = No             677
# 7 = Refused          0
# 9 = Don't know       2
# Missing          4,634
#

EXPECTED_ALQ110 = {
    1: 746,
    2: 677,
    9: 2
}


actual = (
    df["ALQ110"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ110 ---")


if (
    actual != EXPECTED_ALQ110
    or
    int(
        df["ALQ110"]
        .isna()
        .sum()
    )
    != 4634
):

    raise ValueError(
        "ALQ110 differs from CDC."
    )


print(
    "PASS: ALQ110 matches CDC."
)


# ============================================================
# 13. CDC CHECK - ALQ120Q
# ============================================================

#
# CDC:
#
# 0-360 = 4,650
# 777   = 0
# 999   = 2
# Missing = 1,407
#
# CDC explicitly says:
#
#     ENTER '0' FOR NEVER
#
# The attached XPT contains 986 tiny values that
# therefore represent zero.
#

s = df["ALQ120Q"]


print("\n--- ALQ120Q ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-360:",
    int(
        s.between(
            0,
            360
        ).sum()
    )
)


print(
    "Don't know (999):",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 986
    and
    int(
        s.between(
            0,
            360
        ).sum()
    )
    == 4650
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 2
    and
    int(
        s.isna()
        .sum()
    )
    == 1407
):

    raise ValueError(
        "ALQ120Q differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: ALQ120Q matches CDC."
)


# ============================================================
# 14. CDC CHECK - ALQ120U
# ============================================================

#
# CDC:
#
# 1 = Week   1,567
# 2 = Month    926
# 3 = Year   1,171
# Missing   2,395
#

EXPECTED_ALQ120U = {
    1: 1567,
    2: 926,
    3: 1171
}


actual = (
    df["ALQ120U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ120U ---")


if (
    actual != EXPECTED_ALQ120U
    or
    int(
        df["ALQ120U"]
        .isna()
        .sum()
    )
    != 2395
):

    raise ValueError(
        "ALQ120U differs from CDC."
    )


print(
    "PASS: ALQ120U matches CDC."
)


# ============================================================
# 15. CDC CHECK - ALQ130
# ============================================================

#
# CDC:
#
# 1-36 = 3,664
# 777  = 0
# 999  = 2
# Missing = 2,393
#

s = df["ALQ130"]


print("\n--- ALQ130 ---")


print(
    "1-36:",
    int(
        s.between(
            1,
            36
        ).sum()
    )
)


print(
    "Don't know (999):",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            36
        ).sum()
    )
    == 3664
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 2
    and
    int(
        s.isna()
        .sum()
    )
    == 2393
):

    raise ValueError(
        "ALQ130 differs from CDC."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 16. CDC CHECK - ALQ140Q
# ============================================================

#
# CDC:
#
# 0-304 = 3,659
# 777   = 0
# 999   = 7
# Missing = 2,393
#
# CDC explicitly says:
#
#     ENTER '0' FOR NONE
#
# The attached XPT contains 2,159 tiny values that
# therefore represent zero.
#

s = df["ALQ140Q"]


print("\n--- ALQ140Q ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-304:",
    int(
        s.between(
            0,
            304
        ).sum()
    )
)


print(
    "Don't know (999):",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 2159
    and
    int(
        s.between(
            0,
            304
        ).sum()
    )
    == 3659
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 7
    and
    int(
        s.isna()
        .sum()
    )
    == 2393
):

    raise ValueError(
        "ALQ140Q differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: ALQ140Q matches CDC."
)


# ============================================================
# 17. CDC CHECK - ALQ140U
# ============================================================

#
# CDC:
#
# 1 = Week     399
# 2 = Month    306
# 3 = Year     795
# Missing    4,559
#

EXPECTED_ALQ140U = {
    1: 399,
    2: 306,
    3: 795
}


actual = (
    df["ALQ140U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ140U ---")


if (
    actual != EXPECTED_ALQ140U
    or
    int(
        df["ALQ140U"]
        .isna()
        .sum()
    )
    != 4559
):

    raise ValueError(
        "ALQ140U differs from CDC."
    )


print(
    "PASS: ALQ140U matches CDC."
)


# ============================================================
# 18. CDC CHECK - ALQ150
# ============================================================

#
# CDC:
#
# 1 = Yes          878
# 2 = No         3,772
# 7 = Refused        0
# 9 = Don't know     2
# Missing        1,407
#

EXPECTED_ALQ150 = {
    1: 878,
    2: 3772,
    9: 2
}


actual = (
    df["ALQ150"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ150 ---")


if (
    actual != EXPECTED_ALQ150
    or
    int(
        df["ALQ150"]
        .isna()
        .sum()
    )
    != 1407
):

    raise ValueError(
        "ALQ150 differs from CDC."
    )


print(
    "PASS: ALQ150 matches CDC."
)


# ============================================================
# 19. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected numeric changes from the raw pandas XPT:
#
# ALQ120Q:
#     986 tiny values -> 0
#
# ALQ140Q:
#     2,159 tiny values -> 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 9 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# All ALQ_F variables are proven whole-number
# fields in this release.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 21. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 9 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 23. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 24. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 25. RAW CSV .0 FORMATTING CHECK
# ============================================================

#
# ALQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere
# in the final CSV.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "ALQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "ALQ120Q zero restorations:",
    986
)


print(
    "ALQ140Q zero restorations:",
    2159
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)

NHANES 2009-2010 ALQ_F XPT -> CSV
Rows: 6,059
Columns: 9
PASS: Structure = 6,059 rows x 9 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 3,145

Tiny values by variable:
ALQ140Q    2159
ALQ120Q     986

PASS: Tiny-value pattern matches validated ALQ_F pattern.

--- ZERO RESTORATION ---
ALQ120Q: 986 restored to 0
ALQ140Q: 2,159 restored to 0

PASS: 3,145 validated zero values restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in ALQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: ALQ_F questionnaire fields formatted without unnecessary .0.

--- ALQ101 ---
PASS: ALQ101 matches CDC.

--- ALQ110 ---
PASS: ALQ110 matches CDC.

--- ALQ120Q ---
Zero: 986
0-360: 4650
Don't know (999): 2
Missing: 1407
PASS: ALQ120Q matches CDC.

--- ALQ120U ---
PASS: ALQ120U matches CDC.

--- ALQ130 ---
1-36: 3664
Don't know (999): 2
Missing: 2393
PASS: ALQ130 matches CDC.

--- ALQ140Q